<strong><span style="color:skyblue;font-size:40px"> Reinforcement Learning model for Quant analysis</strong><br>
- Data source from yahoo finance<br>
- train and test timer periods<br>
- Gymnasium for the model reward<br>

In [20]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import stable_baselines3 as sb3
import yfinance as yf
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
import os


<strong><span style="color:lightgreen;font-size:30px">Agent environment Build</strong><br>
- Uses OpenAI's gymnasium space<br>
<span style="color:red">- Reward function requires work!</strong>

In [35]:
class StockTradingEnv(gym.Env):

    def __init__(self, data):
        self.data = data
        self.current_step = 0
        self.action_space = spaces.Discrete(3)  # Buy, Hold, Sell
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(data.shape[1],), dtype=np.float32)
        self.cash = 10000
        self.holdings = 0
        self.reset()

    def _get_state(self):
        step = min(self.current_step, len(self.data) - 1)
        return np.array([
        self.data.iloc[self.current_step]['Close'],
        self.cash,
        self.holdings,
        0.0,  # pad if needed
        0.0
        ], dtype=np.float32)
    
    def step(self, action):
        # Reward based on the current action
        reward = self.calculate_reward(action)
        # Advance to the next time step
        self.current_step += 1
        # Check if we’re done (end of the data)
        terminated = self.current_step >= len(self.data) - 1
        truncated = False  # Could be used for time limits, etc.
        # Get the next state observation
        observation = self._get_state()
        # Info dict — can be used for debugging, analytics, etc.
        info = {"portfolio_value": self.cash + self.holdings * self.data.iloc[self.current_step]['Close'],}
        return observation, reward, terminated, truncated, info

    def calculate_reward(self,action):
        """"
            Reward is focused on total portfolio profit, the aim is to make and keep the money rather than make reckless trades.

            reward is based on:
            - change in total portfolio balance
            - penalty for extreme actions (buying/selling all holdings)
        """

        price = self.data.iloc[self.current_step]['Close']
        prev_value = self.cash + self.holdings * price

        # Actions
        if action == 1: #Buy
            if self.cash >= price:
                self.holdings += 1
                self.cash -= price
        if action == 2: #Sell
            if self.holdings > 0:
                self.holdings -= 1
                self.cash += price
        # Else Holdings == No Change

        new_value = self.cash + self.holdings * price
        reward = new_value - prev_value

        # Risk Penalty
        allocation = self.holdings*price/(new_value + 1e-5)  # Avoid division by zero

        if allocation < 0.2 or allocation > 0.8:
            risk_penalty = -1
        else:
            risk_penalty = 0

        return reward + risk_penalty
    
    def reset(self, *, seed=None, options=None):
        self.current_step = 0
        self.cash = 10000
        self.holdings = 0
        if seed is not None:
            np.random.seed(seed)    
        return self._get_state(), {}


Financial moddeling functions

In [ ]:
def rolling_statistics(data, window):
    """
    Calculates rolling statistics given a series of data and window size.
    Returns: Rolling Max/Min.Standard deviation/ Average
    Parameters:
        data: Pandas Series
        window: int, length of rolling window
    """
    rolling_max = data.rolling(window=window).max()
    rolling_min = data.rollling(window=window).min()
    rolling_std = data.rolling(window=window).std()
    rolling_average = data.rolling(wondow=window).mean()
    return rolling_max, rolling_min, rolling_std, rolling_average

def rsi (data,window):
    """
    calcualtes the RSI score (Relative Strength Index) for a given data series
    :paramaters
        data: pandas Series
        Window: int, length of rolling window
    """
    delta = data.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta > 0, 0)
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    rs = avg_gain/(avg_loss + 1e-10)  # Avoid division by zero
    rsi = 100 - (100 / (1+rs))
    return rsi

def macd(data, short_window, long_window, signal_window):
    """
    Moving Average Convergence Divergence (MACD) calculation.
    Meansures trend strength and direction.
    Paramenters:
        data: Pandas Series
        short_window: int, short-term moving average window - recomended = 12
        long_window: int, Long-term moving average window - recomended = 26
        signal_window: int, Signal line moving average window - recomended = 9
    Retuerns:
        MACD_line:Pandas Series
        Signal_line: Pandas Series
    """
    ema_short = data.ewm(span=short_window, adjust=False).mean()
    ema_long = data.ewm(span=long_window, adjust=False).mean()
    data['MACD'] = ema_short - ema_long
    data['MACD_signal'] = data['MACD'].ewn(span=signal_window, adjust=False).mean()


<strong><span style="font-size:30px;color:lightpink">download stocks data from yfinance ready for analysis</strong><br>
| Ticker  | Company Name            |
|---------|------------------------|
| TSLA    | Tesla Inc              |
| NVDA    | NVIDIA Corporation     |
| PG      | Procter & Gamble Co    |
| BARC.L  | Barclays PLC           |
| EZJ.L   | EasyJet PLC            |
| BA.L    | BAE Systems PLC        |

In [ ]:
tickers = ['TSLA', 'NVDA', 'PG', 'BARC.L', 'EZJ.L', 'BA.L']
print("Downloading data stocks data...")
data = yf.download(tickers= tickers, start='2018-01-01', end='2024-12-31', group_by='ticker')
print("stocks data downloaded.")
# reshare the data to be multi idex table
data = data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
# convert to ordered by ticker then date
data = data.sort_values(by=['Ticker', 'Date'])
print(data.dtypes)

training_data = data[data['Date'] < '2022-01-01']
testing_data = data[data['Date'] >= '2022-01-01']

[*********************100%***********************]  6 of 6 completed

stocks data downloaded.
Price
Date      datetime64[ns]
Ticker            object
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object



/var/folders/f2/_cjvdffn2x91f953fzbj08dc0000gn/T/ipykernel_6159/1129170589.py:7: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()


In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']

symbol = 'TSLA'
symbol_data = training_data[training_data['Ticker'] == symbol][features]

log_dir = "/Users/zacwells/Desktop/Reinforcement Learing.stock_trading_env_TSLA_PPO.csv"
os.makedirs(log_dir,exist_ok=True)

env = StockTradingEnv(symbol_data)
check_env(env, warn=True)

model = sb3.PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=2_000_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.01e+03 |
|    ep_rew_mean     | -788     |
| time/              |          |
|    fps             | 6895     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.01e+03    |
|    ep_rew_mean          | -742        |
| time/                   |             |
|    fps                  | 5184        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011632193 |
|    clip_fraction        | 0.0294      |
|    clip_range           | 0.2         |
|    entropy_loss   